# AtendeAI — Fase 1: MVP (chain com saída estruturada)
Projeto de Disciplina — AI Agents com LangChain. Domínio: atendente virtual de pós-venda de e-commerce.

**Uso de IA:** ver `docs/uso_de_ia.md`.

**Evidências desta fase (checklist do enunciado):**
- [ ] Anatomia do agente e classificação arquitetural (seção 1)
- [ ] Esboço de planejamento hierárquico e de memória curto/longo prazo (seção 1)
- [ ] PromptTemplate e ChatPromptTemplate (seção 2)
- [ ] Saída JSON com output parser (seção 2)
- [ ] Runnable: invoke, batch, stream (seção 2)
- [ ] Chain em LCEL e com `@chain` (seções 2 e 3)
- [ ] Padrão adotado, com justificativa técnica (seção 4)

## 1. Anatomia e arquitetura do agente

| Componente | Escolha para o AtendeAI | Observação |
|---|---|---|
| LLM | Modelo de chat com tool calling (OpenAI via `LLM_MODE=openai`); modo dummy para reprodução sem credenciais | `temperature=0` para respostas estáveis |
| System message | Persona de atendente de pós-venda, regras (não inventar política, sempre responder no esquema JSON) | Evolui para PTCF na Fase 2 |
| Memória | Curto prazo: histórico da sessão. Longo prazo: perfil e casos anteriores do cliente | Aqui só o desenho; implementação na Fase 3 |
| Ferramentas | SQL de pedidos (text-to-query), RAG de políticas (FAISS), ViaCEP (API externa), cálculo de prazo/frete (hardcoded) | Fases 2 e 4 |
| Base de conhecimento | 6 documentos de política da loja (`data/politicas/`) indexados em FAISS | Índice tratado como pré-existente |

### Classificação arquitetural
**Padrão: híbrido.** Pedidos simples (status do pedido, prazo para um CEP) seguem um caminho **reativo**: percepção, ferramenta, resposta. Casos compostos (ex.: troca por defeito com cálculo de prazo do novo envio) exigem **deliberar**: decompor a tarefa, decidir a ordem das ferramentas e verificar a política antes de responder. Um roteador direciona cada mensagem ao caminho certo.

**Maturidade agêntica** (escala usada neste relatório, a ajustar ao espectro apresentado em aula):

| Nível | Descrição |
|---|---|
| N1 | Chain fixa com saída estruturada |
| N2 | Agente reativo que escolhe ferramentas |
| N3 | Agente com memória, planejamento e verificação |
| N4 | Sistema multiagente |

Hoje: **N1**. Fim do projeto: **N3**, projetado para ser um nó dentro de um sistema N4 na disciplina seguinte.

### Planejamento hierárquico (esboço)
Exemplo de tarefa composta: *"Comprei um tênis (pedido 10234) há 20 dias, veio com defeito. Quero trocar e saber o prazo de entrega do novo para o CEP 01310-100."*

```
Objetivo: resolver a troca por defeito e informar o prazo do novo envio
├── 1. Identificar pedido e item                    [SQL]
│     ├── 1.1 extrair numero_pedido da mensagem
│     └── 1.2 consultar data, item e status do pedido
├── 2. Verificar elegibilidade                      [RAG]
│     ├── 2.1 buscar a política de defeito e garantia
│     ├── 2.2 comparar os 20 dias com os prazos da política
│     └── 2.3 verificar a resposta contra a evidência (NLI, Fase 5)
├── 3. Estimar prazo e frete do novo envio          [API + hardcoded]
│     ├── 3.1 validar o CEP (ViaCEP)
│     └── 3.2 calcular prazo e frete pela tabela da loja
└── 4. Compor uma única resposta ao cliente
```
Cada folha corresponde a uma ferramenta. As etapas 1.x e 3.1 são independentes e podem rodar em paralelo.

### Memória: curto vs. longo prazo (esboço)
| Tipo | O que guarda no AtendeAI |
|---|---|
| Curto prazo | Turnos da sessão, pedido e CEP em discussão, intenção corrente |
| Longo prazo, semântica | Fatos estáveis do cliente (CEP padrão, canal preferido) |
| Longo prazo, episódica | Casos anteriores (ex.: "trocou tênis em outubro e reclamou do prazo") |
| Longo prazo, procedural | Regras de atendimento e prompt de sistema versionado |

A taxonomia CoALA e a estratégia de janela de contexto são detalhadas na Fase 3.

## 2. Templates, saída estruturada e interface Runnable

In [ ]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from src.llm_factory import get_llm
from src.schemas import AtendimentoOutput

llm = get_llm()
print("Modo:", os.getenv("LLM_MODE", "dummy"), "| modelo:", type(llm).__name__)

### 2.1 `PromptTemplate`: chamada de texto simples

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.exceptions import OutputParserException

resumo_prompt = PromptTemplate.from_template(
    "Resuma em uma frase curta, em português, a solicitação do cliente: {mensagem}"
)
resumo_chain = resumo_prompt | llm | StrOutputParser()
print(resumo_chain.invoke({"mensagem": "Oi, meu pedido 10234 ainda não chegou"}))

### 2.2 `ChatPromptTemplate` + output parser em JSON (Pydantic)

In [ ]:
parser = PydanticOutputParser(pydantic_object=AtendimentoOutput)

SYSTEM = """Você é um atendente de pós-venda de e-commerce.
Analise a mensagem do cliente e responda SOMENTE com um objeto no esquema JSON abaixo, sem texto extra.
- intencao: uma de status_pedido, troca_devolucao, prazo_frete, cancelamento, saudacao, outro
- entidades: numero_pedido, cep, produto (use null quando ausente)
- requer_ferramenta: true se for preciso consultar pedido, política ou CEP para responder
- resposta: resposta curta e cordial; não invente políticas, prazos ou dados
- confianca: número de 0 a 1

{format_instructions}"""

chat_prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM), ("human", "{mensagem}")]
).partial(format_instructions=parser.get_format_instructions())

print(chat_prompt.input_variables)

### 2.3 Chain declarativa (LCEL): `prompt | llm | parser`

In [ ]:
chain_lcel = chat_prompt | llm | parser

exemplo = "Onde está meu pedido 10234? Já faz 9 dias."
r = chain_lcel.invoke({"mensagem": exemplo})          # invoke: chamada única
print(type(r).__name__)
print(r.model_dump_json(indent=2))

**`batch`**: várias mensagens de uma vez, incluindo casos ambíguos do domínio.

In [ ]:
mensagens = [
    "Onde está meu pedido 10234? Já faz 9 dias.",
    "Quero devolver o tênis do pedido #10234, comprei há 5 dias",
    "Qual o prazo de entrega para o CEP 01310-100?",
    "Quero cancelar o pedido 10250",
    "Meu pedido 10234 não chegou, qual o prazo?",   # ambígua: status ou prazo?
    "Vocês vendem cartão presente?",                # fora do escopo
]

t0 = time.perf_counter()
resultados = chain_lcel.batch([{"mensagem": m} for m in mensagens])
print(f"batch de {len(mensagens)} mensagens em {time.perf_counter() - t0:.2f}s")

import pandas as pd
df_lcel = pd.DataFrame([
    {"mensagem": m, "intencao": r.intencao, "pedido": r.entidades.numero_pedido,
     "cep": r.entidades.cep, "ferramenta?": r.requer_ferramenta, "confianca": r.confianca}
    for m, r in zip(mensagens, resultados)
])
df_lcel

**`stream`**: aplicado às etapas `prompt | llm`, que emitem tokens. Para o JSON estruturado completo, streaming de tokens brutos tem pouco valor para o usuário. O ponto onde ele faz sentido é a **resposta final** ao cliente (Fase 5 e interface), e por isso é demonstrado aqui sobre a saída do modelo.

In [ ]:
pedacos = []
for chunk in (chat_prompt | llm).stream({"mensagem": "Qual o prazo para o CEP 01310-100?"}):
    pedacos.append(chunk.content)
print(f"{len(pedacos)} chunks recebidos. Primeiros 5:", pedacos[:5])

**Por que usar um parser estruturado:** texto fora do esquema levanta um erro previsível e tratável, em vez de quebrar o resto do sistema.

In [ ]:
try:
    parser.parse("isto não é JSON")
except OutputParserException as e:
    print("Erro capturado:", str(e)[:150], "...")

## 3. Composição imperativa com `@chain`

In [ ]:
from langchain_core.runnables import chain

@chain
def analisar_mensagem(entrada: dict) -> AtendimentoOutput:
    """Mesma lógica da chain LCEL, com controle explícito de fluxo:
    validação da entrada, até 2 tentativas de parsing e fallback seguro."""
    mensagem = (entrada.get("mensagem") or "").strip()
    if not mensagem:
        return AtendimentoOutput(
            intencao="outro", entidades={}, requer_ferramenta=False,
            resposta="Pode me contar como posso ajudar?", confianca=1.0,
        )
    prompt_value = chat_prompt.invoke({"mensagem": mensagem})
    for tentativa in range(2):
        ai_msg = llm.invoke(prompt_value)
        try:
            return parser.invoke(ai_msg)
        except OutputParserException:
            if tentativa == 1:
                return AtendimentoOutput(
                    intencao="outro", entidades={}, requer_ferramenta=False,
                    resposta="Não consegui interpretar sua mensagem. Pode reformular?", confianca=0.0,
                )

print(analisar_mensagem.invoke({"mensagem": exemplo}).model_dump())
print("mensagem vazia ->", analisar_mensagem.invoke({"mensagem": "  "}).resposta)

### 3.1 As duas composições produzem o mesmo resultado?

In [ ]:
resultados_chain = analisar_mensagem.batch([{"mensagem": m} for m in mensagens])

linhas = []
for m, a, b in zip(mensagens, resultados, resultados_chain):
    linhas.append({"mensagem": m, "LCEL": a.intencao, "@chain": b.intencao,
                   "mesma intenção": a.intencao == b.intencao,
                   "mesmas entidades": a.entidades == b.entidades})
pd.DataFrame(linhas)

Com um LLM real, pequenas diferenças são esperadas mesmo com `temperature=0`. O que importa é a intenção e as entidades coincidirem nos casos claros.

### 3.2 Diferença observável: streaming

In [ ]:
chunks_lcel = list((chat_prompt | llm).stream({"mensagem": exemplo}))
chunks_chain = list(analisar_mensagem.stream({"mensagem": exemplo}))
print("LCEL (prompt | llm): ", len(chunks_lcel), "chunks")
print("@chain:              ", len(chunks_chain), "chunk(s)")
print("Etapas da chain LCEL:", [type(s).__name__ for s in chain_lcel.steps])

## 4. Decisão: padrão de composição do agente

| Critério | LCEL (pipe) | `@chain` (imperativo) |
|---|---|---|
| Streaming | Os chunks do modelo passam pelas etapas (visto em 3.2) | A função devolve tudo ao final: 1 chunk (visto em 3.2) |
| Retry e fallback | Declarativos: `.with_retry()`, `.with_fallbacks()` | Manuais, no corpo da função (como em 3) |
| Lógica condicional e laços | Exige `RunnableBranch` ou `RunnableLambda` | Natural em Python |
| Estrutura | Grafo de etapas inspecionável (`chain_lcel.steps`) | Função opaca para o grafo |
| Observabilidade no LangSmith | Cada etapa é um run | A função é um run; chamadas internas devem aparecer como filhas (**a confirmar nos traces da Fase 5**) |
| Integração com as próximas fases | `create_openai_tools_agent` devolve um Runnable LCEL, e `RunnableWithMessageHistory` e `AgentExecutor` compõem com Runnables | Também é um Runnable, então encaixa, mas sem as vantagens acima |

**Decisão: LCEL é o padrão do AtendeAI.** O agente final será montado de componentes Runnable (prompt, modelo, parser, ferramentas, histórico), e a composição declarativa preserva streaming de ponta a ponta, retry e fallback configuráveis e um grafo inspecionável, que ajudam a depurar o agente nas fases seguintes. **`@chain` fica como exceção controlada**: apenas para trechos com lógica condicional ou laços que seriam artificiais em LCEL (como o reparo de parsing acima), sempre embrulhado como uma etapa dentro de uma chain LCEL.

**Risco aceito:** LCEL fica menos legível quando a lógica cresce. Mitigação: nomear etapas com `.with_config(run_name=...)` e usar callbacks (Fase 2).

## 5. Resumo e próximos passos
- Chain ponta a ponta (prompt estruturado, modelo, parser JSON) funcionando nos dois estilos.
- Padrão adotado: LCEL, com `@chain` como exceção.
- **Para o relatório:** tirar screenshots das saídas das seções 2.3, 3.1 e 3.2 e anotar o modo usado (dummy ou real).
- **Fase 2:** agente ReAct (`initialize_agent`), ferramentas (incluindo ViaCEP), PTCF, `RouterChain`, parser com tratamento de erro, memória e callbacks.